# Python RAG 交互式学习笔记

这份 Notebook 是 `python_rag_notes.md` 的配套运行版。它用于复习 RAG 的核心链路：数据加载、文本切片、Embedding 思路、向量检索、Prompt 组装和答案生成。

使用建议：

- 前半部分全部离线可运行，不需要 API key，也不连接外部数据库。
- Milvus、真实 Embedding 和 LLM 调用只保留为扩展方向，具体可参考 `python_milvus_notes.ipynb`。
- 学习时先跑离线示例，理解流程后再替换为真实向量库和模型。

## 1. RAG 的主流程

RAG 是 Retrieval-Augmented Generation：先检索，再生成。

```text
数据准备：文件 -> Loader -> 清洗 -> Splitter -> Chunks -> Embedding -> VectorStore
查询生成：问题 -> Query Embedding -> 检索 -> 相关片段 -> Prompt -> LLM -> Answer
```

RAG 的关键不是“把资料塞给模型”，而是把正确资料以合适粒度取回来，并让模型严格基于资料回答。

In [ ]:
from dataclasses import dataclass


@dataclass
class SimpleDocument:
    page_content: str
    metadata: dict


raw_documents = [
    SimpleDocument(
        "RAG 会先检索知识库，再把相关上下文交给大语言模型回答。",
        {"source": "rag_intro.md", "title": "RAG 原理"},
    ),
    SimpleDocument(
        "Milvus 是向量数据库，常用于语义搜索、相似度检索和企业知识库。",
        {"source": "milvus.md", "title": "Milvus"},
    ),
    SimpleDocument(
        "Text Splitter 会把长文档切成多个文本块，chunk_size 和 overlap 会影响召回质量。",
        {"source": "splitter.md", "title": "文本切片"},
    ),
]

raw_documents

## 2. 文本切片

切片要平衡两个目标：

- 太短：语义不完整，检索回来缺上下文。
- 太长：噪声太多，Prompt 成本高，也可能淹没答案。

常见做法是先按标题、段落等结构切，再按长度兜底。

In [ ]:
def simple_split_text(text: str, chunk_size: int = 32, overlap: int = 8) -> list[str]:
    if chunk_size <= overlap:
        raise ValueError("chunk_size 必须大于 overlap")
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        if end >= len(text):
            break
        start = end - overlap
    return chunks


long_text = (
    "RAG 项目通常先加载 PDF、Word、Markdown 等文档，然后清洗文本，"
    "再按照标题、段落或固定长度切成文本块。文本块会被 Embedding 模型转成向量，"
    "写入 Milvus、Chroma 或 FAISS 等向量库。用户提问时，系统先检索相关文本块，"
    "再让模型根据上下文生成答案。"
)

chunks = simple_split_text(long_text, chunk_size=45, overlap=10)
for i, chunk in enumerate(chunks, 1):
    print(i, chunk)

## 3. 用关键词重叠模拟检索

真实 RAG 会用 Embedding 向量做相似度检索。为了离线理解流程，这里用字符重叠模拟一个很小的 retriever。

In [ ]:
chunk_docs = [
    SimpleDocument(chunk, {"source": "rag_pipeline.md", "chunk": i})
    for i, chunk in enumerate(chunks, 1)
]
chunk_docs.extend(raw_documents)


def keyword_retrieve(question: str, documents: list[SimpleDocument], k: int = 3) -> list[tuple[int, SimpleDocument]]:
    query_terms = set(question.lower())
    scored = []
    for doc in documents:
        score = len(query_terms & set(doc.page_content.lower()))
        scored.append((score, doc))
    scored.sort(key=lambda item: item[0], reverse=True)
    return [(score, doc) for score, doc in scored[:k] if score > 0]


question = "RAG 为什么需要切片和向量库？"
for score, doc in keyword_retrieve(question, chunk_docs):
    print(score, doc.metadata, doc.page_content)

## 4. 组装 RAG Prompt

Prompt 中应明确三件事：

1. 只能根据上下文回答。
2. 上下文没有答案时说不知道。
3. 尽量返回来源，便于追溯。

In [ ]:
def build_rag_prompt(question: str, retrieved: list[tuple[int, SimpleDocument]]) -> str:
    context = []
    for score, doc in retrieved:
        source = doc.metadata.get("source", "unknown")
        chunk = doc.metadata.get("chunk", "-")
        context.append(f"[source={source}, chunk={chunk}, score={score}] {doc.page_content}")

    return "\n".join(
        [
            "你是一个严谨的 RAG 助手。",
            "请只根据给定上下文回答；如果上下文没有答案，就说不知道。",
            "",
            "上下文：",
            *context,
            "",
            f"问题：{question}",
        ]
    )


retrieved = keyword_retrieve(question, chunk_docs)
print(build_rag_prompt(question, retrieved))

## 5. 模拟生成答案

真实项目中，最后一步会调用 LLM。这里为了离线运行，用一个简单函数模拟“基于上下文回答”。

In [ ]:
def fake_llm_answer(question: str, retrieved: list[tuple[int, SimpleDocument]]) -> str:
    if not retrieved:
        return "不知道。当前知识库没有检索到足够相关的上下文。"
    sources = ", ".join(str(doc.metadata.get("source")) for _, doc in retrieved)
    return (
        "RAG 需要切片，是为了把长文档拆成适合检索和放入 Prompt 的文本块；"
        "需要向量库，是为了保存文本块向量并根据用户问题做相似度检索。"
        f"\n来源：{sources}"
    )


print(fake_llm_answer(question, retrieved))

## 6. 可选：LangChain Text Splitter

如果已安装 `langchain-text-splitters`，可以使用 `RecursiveCharacterTextSplitter`。它会按分隔符优先级递归切割，比手写固定长度更适合学习项目。

In [ ]:
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=60,
        chunk_overlap=12,
        separators=["\n\n", "\n", "。", "，", ""],
    )
    lc_chunks = splitter.split_text(long_text)
    for i, chunk in enumerate(lc_chunks, 1):
        print(i, chunk)
except ImportError:
    print("未安装 langchain-text-splitters，跳过 LangChain splitter 示例。")

## 7. Milvus RAG 实战位置

Milvus 在 RAG 中通常作为 VectorStore：

```text
Document chunks -> Embedding -> Milvus collection
User query -> Query Embedding -> Milvus search -> Prompt context
```

完整的 Milvus 运行、collection、schema、索引、过滤搜索、LangChain Milvus Retriever 示例，放在 `python_milvus_notes.ipynb`。

## 8. 高级 RAG 复习清单

| 方向 | 核心用途 |
| --- | --- |
| Query Rewrite | 改写用户问题，提高召回 |
| Multi Query | 多角度生成查询，合并结果 |
| Hybrid Search | 稠密向量 + BM25 / 稀疏向量 |
| Rerank | 对候选片段重新排序 |
| Corrective RAG | 判断检索质量，不足时修正 |
| Adaptive RAG | 按问题复杂度选择策略 |
| Agentic RAG | 由 Agent 决定是否检索、调用工具或追问 |
| Multimodal RAG | 文本、图片、表格等多模态资料联合检索 |


## 9. 什么样的 RAG 项目更吸引面试官

面试官通常不只看“能不能问答”，而是看你有没有解决真实 RAG 工程问题。

| 层级 | 可讲亮点 |
| --- | --- |
| 数据解析 | PDF、图片、PPT、表格、公式、OCR、版面分析 |
| 切片策略 | 标题递归切片、语义切片、Parent-Child、保留标题路径 |
| 检索策略 | Dense + BM25、Hybrid Search、metadata filter、score threshold |
| 重排策略 | BGE/Qwen reranker、cross-encoder 精排 |
| 生成策略 | 引用来源、拒答、结构化输出、上下文压缩 |
| 评估体系 | Recall@K、MRR、NDCG、Faithfulness、端到端准确率 |
| 工程化 | 多租户权限、缓存、异步入库、日志追踪、成本控制 |

一句话模板：

```text
+我做的不是简单 PDF 问答，而是一个带解析、切片、混合检索、重排、引用和评估闭环的 RAG 系统。
+```

## 10. 上下文增强 Contextual Retrieval

传统 RAG 的常见问题：

1. 固定切片导致上下文丢失，chunk 离开原文后语义漂移。
2. 文档名、章节、年份、产品型号等关键元信息没有进入 chunk，导致检索失败。
3. 单一向量检索不擅长精确关键词，单一 BM25 又不懂语义。

上下文增强的做法：

```text
原始文档 + 当前 chunk
-> 为 chunk 生成上下文描述
-> 上下文描述 + chunk 正文
-> embedding / BM25 / metadata index
```

实际项目里可以给每个 chunk 增加：文档标题、章节路径、页码、主题摘要、关键实体、适用时间范围。这样检索时更容易命中，生成时也更少误解。

## 11. Agentic RAG

Agentic RAG 可以理解为：

```text
RAG + Agent = Agentic RAG
```

传统 RAG 是线性流程：检索一次，生成一次。Agentic RAG 让 Agent 参与检索决策：

- 是否需要检索。
- 是否需要改写 query。
- 是否拆成多个子问题。
- 选择向量库、BM25、图谱、Web、SQL、API 中的哪个工具。
- 检索结果是否足够。
- 不足时继续检索、换策略或追问。

适合多跳问题、多知识源问题、实时工具问题和需要答案自检的问题。不适合简单 FAQ 或极低延迟场景。

## 12. 多模态 RAG

多模态 RAG 处理的是 PDF、扫描件、图片、PPT、表格、图表、音频等资料。

```text
PDF / 图片 / PPT
-> OCR / Layout Parser / 多模态模型
-> 文本、图片、表格、公式结构化
-> 文本 embedding + 图像 embedding / 多模态 embedding
-> Milvus 多向量字段或多 collection
-> hybrid search + rerank
-> 多模态 LLM 生成答案
```

常见方向：

| 类型 | 例子 |
| --- | --- |
| OCR / 文档解析 | PaddleOCR、Dolphin、Dots.OCR、DeepSeek-OCR |
| 多模态 embedding | GME-Qwen2-VL、CLIP、ColPali、ColQwen |
| Reranker | BGE Reranker、Qwen Reranker、Cohere Rerank |
| 多模态推理 | Qwen-VL、GLM-4V、GPT-4o/4.1 视觉模型 |

## 13. Reranker 与两阶段检索

RAG 常见两阶段检索：

```text
第一阶段：向量/BM25 快速召回 top 20-100
第二阶段：reranker 对 query-document pair 精排，选 top 3-8 给 LLM
```

Embedding 检索速度快，适合大规模召回；Reranker 更慢，但能更细地判断“这个片段是否真的回答了这个问题”。

需要 reranker 的典型情况：

- top 20 里有答案，但 top 3 不稳定。
- 文档相似段落很多。
- 问题包含多个条件。
- hybrid search 之后需要统一排序。

## 14. RAG 评估指标

| 指标 | 衡量什么 |
| --- | --- |
| Recall@K | 正确片段是否进入前 K 个结果 |
| Precision@K | 前 K 个结果里相关片段比例 |
| MRR | 第一个正确结果排得多靠前 |
| NDCG | 多个相关结果的排序质量 |
| Faithfulness | 回答是否被上下文支持 |
| Answer Correctness | 最终答案是否正确 |
| Citation Accuracy | 引用是否真的支持答案 |
| Latency / Cost | 延迟和成本 |

面试里要强调：检索评估和生成评估要分开做。先证明召回命中，再证明模型基于命中的上下文答对。